# 03 - Feature Engineering

Compute phenological metrics from NDVI time-series, aggregate weather, create interaction terms, and PCA.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.data_loader import load_merged_data
from src.features import (
    compute_phenological_metrics,
    aggregate_weather_features,
    create_interaction_terms,
    pca_on_spectral_bands,
    build_feature_matrix
)
from src.utils import get_data_dir

try:
    df = load_merged_data()
except FileNotFoundError:
    from src.synthetic_data import generate_synthetic_dataset
    df = generate_synthetic_dataset()
    get_data_dir().mkdir(parents=True, exist_ok=True)
    df.to_csv(get_data_dir() / 'merged_data.csv', index=False)

df['date'] = pd.to_datetime(df['date'])

## 1. Phenological Metrics

In [2]:
pheno = compute_phenological_metrics(df)
print('Phenological metrics:')
print(pheno.describe())

Phenological metrics:
        ndvi_peak  ndvi_peak_doy     sos_doy     eos_doy  season_length  \
count  100.000000     100.000000  100.000000   48.000000      44.000000   
mean     0.768383     156.600000  107.000000  196.208333      77.045455   
std      0.077321       9.025217   29.198035    6.519882      43.056455   
min      0.586624     131.000000   91.000000  171.000000     -10.000000   
25%      0.711726     151.000000   91.000000  191.000000      80.000000   
50%      0.770178     161.000000  101.000000  201.000000      90.000000   
75%      0.819603     161.000000  111.000000  201.000000     110.000000   
max      0.950000     181.000000  201.000000  201.000000     110.000000   

       cumulative_ndvi  ndvi_grain_fill_slope  
count       100.000000             100.000000  
mean         53.257133              -0.010569  
std           3.308749               0.004885  
min          45.245753              -0.022762  
25%          50.982828              -0.013680  
50%          5

## 2. Weather Aggregates

In [3]:
weather = aggregate_weather_features(df)
print(weather.head())

   field_id  precip_sum  temp_mean    solar_sum     gdd_sum
0   field_0  176.577315  17.597966  1496.884615  163.773559
1   field_1  183.423868  19.777184  1612.057244  192.103397
2  field_10  234.204223  18.869784  1530.441827  180.307188
3  field_11  264.101653  18.081663  1510.936719  170.061613
4  field_12  186.977603  18.836175  1585.645483  179.870271


## 3. Interaction Terms & PCA

In [4]:
X_df, y = build_feature_matrix(df, include_pca=True, include_interactions=True)
feature_cols = [c for c in X_df.columns if c != 'field_id']
print(f'Features: {feature_cols}')
X_df.head()

Features: ['ndvi_peak', 'season_length', 'cumulative_ndvi', 'ndvi_grain_fill_slope', 'precip_sum', 'temp_mean', 'solar_sum', 'gdd_sum', 'soil_pH', 'soil_OC', 'soil_clay', 'B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'pca_0', 'pca_1', 'pca_2', 'ndvi_peak_x_precip_sum', 'ndvi_peak_x_gdd_sum', 'ndvi_peak_x_soil_OC', 'gdd_sum_x_precip_sum']


,field_id,ndvi_peak,season_length,cumulative_ndvi,ndvi_grain_fill_slope,precip_sum,temp_mean,solar_sum,gdd_sum,soil_pH,...,B8,B11,B12,pca_0,pca_1,pca_2,ndvi_peak_x_precip_sum,ndvi_peak_x_gdd_sum,ndvi_peak_x_soil_OC,gdd_sum_x_precip_sum
0,field_0,0.778347,100.0,53.944920,-0.010844,176.577315,17.597966,1496.884615,163.773559,6.374540,...,0.356724,0.245738,0.179209,-0.000018,-0.001717,0.005322,137.438404,127.472641,0.815630,28918.695251
1,field_1,0.778501,110.0,57.016537,-0.007646,183.423868,19.777184,1612.057244,192.103397,6.950714,...,0.369398,0.248095,0.181287,0.015104,-0.001583,0.005343,142.795670,149.552692,1.959950,35236.348101
2,field_10,0.858213,NaN,57.634991,-0.013663,234.204223,18.869784,1530.441827,180.307188,6.020584,...,0.380048,0.245901,0.187081,0.027700,-0.000892,-0.000447,200.997108,154.741972,1.298657,42228.704832
3,field_11,0.671525,NaN,50.874385,-0.003571,264.101653,18.081663,1510.936719,170.061613,6.969910,...,0.344838,0.242562,0.186943,-0.013091,0.004932,-0.001456,177.350868,114.200628,0.848615,44913.553166
4,field_12,0.665036,-10.0,50.984927,-0.011008,186.977603,18.836175,1585.645483,179.870271,6.832443,...,0.347514,0.242874,0.183659,-0.010418,0.002082,0.000644,124.346847,119.620215,3.313492,33631.712129


In [5]:
# Save features for modeling
out = X_df.copy()
out['yield'] = y.values
out.to_csv(get_data_dir() / 'features.csv', index=False)
print(f'Saved features to {get_data_dir() / "features.csv"}')

Saved features to /Users/monurajj/Desktop/Satellite-Based Precision Agriculture/data/features.csv
